# Medical Necessity — Question and Answer Review

Takes the question and answer pairs extracted from order form text by
`tnet_clinical_qa_extract` and classifies each pair against the medical necessity criteria.

Input is one payload per trip: `{TripRequestId, RequestDate, qa: [{question, answer}]}`.

Output is one pandas frame per trip, keyed by trip identifier, with a row for every question and
answer pair, a flag for whether the pair is non-medical-necessity, and a confidence level. A
combined frame across all trips is produced alongside.

Classification and confidence both come from the model. It assigns a category, cites the
criterion it relied on, quotes the words it relied on, and rates its own confidence against a
rubric stated in the instruction. No keyword matching is used anywhere in this notebook.

## 1. Configuration

In [ ]:
import os
import re
import json
import glob
import shutil
from datetime import datetime

import pandas as pd

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

KNOWLEDGE_PATH = "/Workspace/Users/josh.smitherman@gmr.net/med_nec/med_nec_knowledge.json"
QA_TABLE = "`prod-sandbox`.vivekkumar_patel.tnet_clinical_qa"
JSON_INPUT_DIR = os.path.join(os.getcwd(), "json_payloads")
OUTPUT_DIR = "/Workspace/Users/josh.smitherman@gmr.net/med_nec/data"
LOCAL_DIR = "/tmp"

SOURCE = "table"

WORKSPACE_BASE_URL = "https://adb-2790612761746757.17.azuredatabricks.net/serving-endpoints"
LLM_MODEL = "databricks-gpt-oss-120b"
LLM_TEMPERATURE = 0.0
LLM_MAX_TOKENS = 4000
LLM_RETRIES = 3
MAX_WORKERS = 8

TRIP_LIMIT = 200
SAMPLE_SEED = 42

CATEGORIES = ["supports_necessity", "insufficient_alone", "non_covered", "administrative", "unclear"]
NON_MED_NEC_CATEGORIES = {"insufficient_alone", "non_covered", "administrative"}
CONFIDENCE_LEVELS = ["High", "Medium", "Low"]

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 140)

print("run:", RUN_TS)
print("input source:", SOURCE)
print("trip limit:", TRIP_LIMIT)

## 2. Criteria

The same reference file the order review notebook uses. Concepts, presumed conditions, bed
confinement, non-covered situations, the intake survey answers and the policy rules all come
from here.

In [ ]:
with open(KNOWLEDGE_PATH, "r") as f:
    KB = json.load(f)

CONCEPTS = {c["concept"]: c for c in KB["concepts"]}
EXCLUSIONS = {e["exclusion_id"]: e for e in KB["exclusions"]}
PRESUMED = {p["id"]: p for p in KB["presumed_necessary"]}
BED = KB["bed_confinement"]
BED_ELEMENTS = {e["element"]: e for e in BED["elements"]}
SURVEY = KB.get("intake_survey", {"sections": []})
SURVEY_ITEMS = {i["id"]: i for sec in SURVEY["sections"] for i in sec["items"]}

CRITERION_IDS = (sorted(CONCEPTS) + sorted(PRESUMED) + sorted(BED_ELEMENTS)
                 + sorted(EXCLUSIONS) + sorted(SURVEY_ITEMS))

CRITERION_SET = set(CRITERION_IDS)

print("knowledge version:", KB["knowledge_version"])
print("concepts:", len(CONCEPTS), "| presumed:", len(PRESUMED),
      "| bed elements:", len(BED_ELEMENTS), "| non-covered:", len(EXCLUSIONS),
      "| survey items:", len(SURVEY_ITEMS))
print("distinct criterion identifiers the model may cite:", len(CRITERION_SET))

## 3. Load the trip payloads

Reads from the output table by default. Set `SOURCE` to `files` in section 1 to read the JSON
payloads written to disk instead.

In [ ]:
trips = []

if SOURCE == "table":
    pdf = (spark.table(QA_TABLE)
           .select("TripRequestId", "RequestDate", "qa_count", "qa_json")
           .orderBy("RequestDate", ascending=False)
           .limit(TRIP_LIMIT)
           .toPandas())
    for r in pdf.itertuples():
        payload = json.loads(r.qa_json)
        trips.append({"trip_id": str(payload.get("TripRequestId", r.TripRequestId)),
                      "request_date": str(payload.get("RequestDate", r.RequestDate)),
                      "qa": payload.get("qa", []) or []})
else:
    paths = sorted(glob.glob(os.path.join(JSON_INPUT_DIR, "*.json")))[:TRIP_LIMIT]
    for path in paths:
        with open(path, "r") as f:
            payload = json.load(f)
        trips.append({"trip_id": str(payload.get("TripRequestId", os.path.basename(path)[:-5])),
                      "request_date": str(payload.get("RequestDate", "")),
                      "qa": payload.get("qa", []) or []})

pair_counts = [len(t["qa"]) for t in trips]
print("trips loaded:", len(trips))
print("question and answer pairs:", f"{sum(pair_counts):,}")
if pair_counts:
    print("pairs per trip: min", min(pair_counts), "| median",
          int(pd.Series(pair_counts).median()), "| max", max(pair_counts))
print()
if trips:
    first = trips[0]
    print("first trip:", first["trip_id"], "|", first["request_date"])
    print(pd.DataFrame(first["qa"]).head(10).to_string(index=False))

## 4. Classification prompt

Compiled from the criteria file. The model receives every question and answer pair for a trip in
one call, so it can see that bed confinement needs all three elements, or that oxygen appears
without a regulation need, rather than judging each pair in isolation.

The model assigns a category, cites the criterion identifier it relied on, quotes the words it
relied on, and rates its own confidence against the rubric below. Nothing in this notebook
matches keywords.

In [ ]:
def compile_prompt(kb):
    concept_lines = [f'  {c["concept"]} | {c["axis"]} axis | {c["status"]} | phrasing: '
                     f'{", ".join(c["terms"][:6])}'
                     for c in sorted(kb["concepts"], key=lambda x: x["concept"])]
    presumed_lines = [f'  {p["id"]} | {p["statement"]}' for p in kb["presumed_necessary"]]
    bed_lines = [f'  {e["element"]} | {e["statement"]}' for e in kb["bed_confinement"]["elements"]]
    exclusion_lines = [f'  {e["exclusion_id"]} | {e["statement"]}' for e in kb["exclusions"]]
    survey_lines = [f'  {i["id"]} | {i["answer"]}'
                    for sec in kb.get("intake_survey", {"sections": []})["sections"]
                    for i in sec["items"]]
    rule_lines = [f'  - {r["statement"]}' for r in kb["policy_rules"]]

    prompt = f"""
You classify question and answer pairs taken from an ambulance transport order form. You judge
only what the pair states. You do not infer, you do not assume, and you handle negation.

The governing question is: {kb["determination_question"]}
{kb["evaluation_principle"]}

CATEGORIES. Assign exactly one to every pair.
  supports_necessity   The answer documents a condition or care need that counts toward medical
                       necessity.
  insufficient_alone   The answer documents something relevant that guidance says does not on its
                       own establish medical necessity, such as bed confinement, or a criterion
                       that requires other elements alongside it.
  non_covered          The answer describes a situation guidance names as not establishing
                       medical necessity, or an operational rather than clinical reason.
  administrative       The pair is logistics, scheduling, billing, trip type, equipment
                       preference or demographics. It says nothing about the patient's condition.
  unclear              The answer is too vague or too short to place, or contradicts itself.

CRITERIA THAT SUPPORT NECESSITY:
{chr(10).join(concept_lines)}

CONDITIONS FOR WHICH NECESSITY MAY BE PRESUMED:
{chr(10).join(presumed_lines)}

BED CONFINEMENT ELEMENTS. All three are required together. Any one alone is insufficient_alone.
{chr(10).join(bed_lines)}

SITUATIONS THAT DO NOT ESTABLISH NECESSITY:
{chr(10).join(exclusion_lines)}

INTAKE SURVEY ANSWERS, for matching form wording to a criterion:
{chr(10).join(survey_lines)}

RULES THAT GOVERN CLASSIFICATION:
{chr(10).join(rule_lines)}

These phrases are non-specific and never establish anything on their own:
{", ".join(kb["non_specific_phrases"])}

You will receive every pair from one trip at once, numbered from zero. Read them together. If a
criterion requires elements that appear in other pairs, take that into account.

OUTPUT valid JSON only, no prose and no code fences, one entry per pair, all keys present:
{{
  "items": [
    {{
      "index": integer,
      "category": "supports_necessity" | "insufficient_alone" | "non_covered" | "administrative" | "unclear",
      "criterion_id": string,
      "evidence": string,
      "reason": string,
      "confidence": "High" | "Medium" | "Low"
    }}
  ]
}}

"criterion_id" is the identifier from the lists above that the classification rests on, or an
empty string for administrative and unclear.
"evidence" is copied verbatim from the question or answer text of that pair, or an empty string
for administrative and unclear.
"reason" is one sentence. Do not state a coverage decision for the trip.

"confidence" is your confidence in the classification of that one pair, judged against this
rubric. It is about the pair, not about the trip.
  High     The pair states the criterion explicitly, in wording that leaves no room for a
           different reading, and the evidence you quoted carries the whole classification.
  Medium   The pair plainly indicates the criterion but uses different wording than the
           criteria list, or needs one short step of reading to connect them.
  Low      The wording is abbreviated, ambiguous, contradicts itself, could reasonably fit
           more than one category, or you could not quote anything that settles it.
Use Low for every pair you place in "unclear".

Return one entry for every pair you are given, in the same numbering."""
    return prompt


QA_PROMPT = compile_prompt(KB)
print(QA_PROMPT[:1600])
print("...")
print("prompt characters:", len(QA_PROMPT))

## 5. Model client

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    if "dbutils" in dir() else os.environ.get("DATABRICKS_TOKEN", "")
)
client = OpenAI(api_key=DATABRICKS_TOKEN or "token-not-set", base_url=WORKSPACE_BASE_URL)


def llm_call(system_prompt, user_prompt, max_tokens=LLM_MAX_TOKENS):
    last_error = None
    for attempt in range(LLM_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "system", "content": system_prompt},
                          {"role": "user", "content": user_prompt}],
                temperature=LLM_TEMPERATURE, max_tokens=max_tokens)
            content = resp.choices[0].message.content
            if isinstance(content, list):
                for item in content:
                    if isinstance(item, dict) and item.get("type") == "text":
                        return item.get("text", "")
                return json.dumps(content)
            return content
        except Exception as e:
            last_error = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"model call failed after {LLM_RETRIES} attempts: {last_error}")


LLM_AVAILABLE = bool(DATABRICKS_TOKEN)
print("token present:", LLM_AVAILABLE, "| model:", LLM_MODEL)

## 6. Validator

A response is accepted only when it parses, returns one entry per pair, and uses a category, a
confidence level and a criterion identifier that exist in the criteria file. An unknown
identifier is blanked rather than trusted, and an unrecognised confidence value falls to Low
rather than being assumed.

In [ ]:
def strip_fences(raw):
    txt = str(raw).strip()
    if txt.startswith("```"):
        txt = re.sub(r"^```[a-zA-Z]*\s*", "", txt)
        txt = re.sub(r"\s*```$", "", txt)
    start, end = txt.find("{"), txt.rfind("}")
    return txt[start:end + 1] if start != -1 and end > start else txt


def valid_json(raw, n_pairs):
    try:
        obj = json.loads(strip_fences(raw))
    except Exception:
        return None, "unparseable"
    items = obj.get("items") if isinstance(obj, dict) else None
    if not isinstance(items, list):
        return None, "missing items array"

    out = {}
    for it in items:
        if not isinstance(it, dict):
            continue
        try:
            idx = int(it.get("index", -1))
        except Exception:
            continue
        if idx < 0 or idx >= n_pairs:
            continue
        category = str(it.get("category", "")).strip()
        if category not in CATEGORIES:
            category = "unclear"
        criterion = str(it.get("criterion_id", "") or "").strip()
        if criterion and criterion not in CRITERION_SET:
            criterion = ""
        confidence = str(it.get("confidence", "") or "").strip().title()
        if confidence not in CONFIDENCE_LEVELS:
            confidence = "Low"
        out[idx] = {"category": category, "criterion_id": criterion,
                    "evidence": str(it.get("evidence", "") or "")[:400],
                    "reason": str(it.get("reason", "") or "")[:400],
                    "confidence": confidence}

    missing = [i for i in range(n_pairs) if i not in out]
    if missing:
        return None, f"missing {len(missing)} of {n_pairs} entries"
    return out, ""


def classify_trip(trip):
    pairs = trip["qa"]
    if not pairs:
        return {"trip_id": trip["trip_id"], "ok": True, "result": {}, "error": "no pairs"}
    lines = [f'{i}. QUESTION: {p.get("question","")}\n   ANSWER: {p.get("answer","")}'
             for i, p in enumerate(pairs)]
    user_prompt = (f"TripRequestId: {trip['trip_id']}\n"
                   f"Pairs to classify: {len(pairs)}\n\n" + "\n".join(lines))
    if not LLM_AVAILABLE:
        return {"trip_id": trip["trip_id"], "ok": False, "result": {}, "error": "no token"}
    try:
        result, err = valid_json(llm_call(QA_PROMPT, user_prompt), len(pairs))
        if result is None:
            retry = llm_call(QA_PROMPT, user_prompt + f"\n\nA previous response was rejected "
                                                      f"({err}). Return one entry for each of the "
                                                      f"{len(pairs)} pairs, numbered from zero.")
            result, err = valid_json(retry, len(pairs))
        if result is None:
            return {"trip_id": trip["trip_id"], "ok": False, "result": {}, "error": err}
        return {"trip_id": trip["trip_id"], "ok": True, "result": result, "error": ""}
    except Exception as e:
        return {"trip_id": trip["trip_id"], "ok": False, "result": {}, "error": str(e)[:300]}

## 7. Classify every trip

In [ ]:
results = {}
if LLM_AVAILABLE and trips:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(classify_trip, t) for t in trips]
        for fut in as_completed(futures):
            r = fut.result()
            results[r["trip_id"]] = r
else:
    for t in trips:
        results[t["trip_id"]] = classify_trip(t)

ok_trips = [t for t in results.values() if t["ok"]]
failed = [t for t in results.values() if not t["ok"]]
print(f"Classified: {len(ok_trips)} of {len(results)} trips "
      f"({len(ok_trips) / max(len(results), 1) * 100:.1f}%)")
if failed:
    print()
    print("failures")
    print(pd.DataFrame([{"trip_id": t["trip_id"], "error": t["error"]}
                        for t in failed]).head(15).to_string(index=False))

## 8. One frame per trip

`trip_frames` is keyed by trip identifier. Every frame carries one row per question and answer
pair, the non-medical-necessity flag, and the confidence level.

In [ ]:
trip_frames = {}
rows = []

for trip in trips:
    tid = trip["trip_id"]
    res = results.get(tid, {"ok": False, "result": {}, "error": "not run"})
    frame_rows = []
    for i, pair in enumerate(trip["qa"]):
        q = str(pair.get("question", ""))
        a = str(pair.get("answer", ""))
        cls = res["result"].get(i) if res["ok"] else None
        if cls is None:
            category, criterion, evidence, reason, level = "unclear", "", "", (
                "Classification failed for this trip, so the pair was not categorised."), "Low"
        else:
            category = cls["category"]
            criterion = cls["criterion_id"]
            evidence = cls["evidence"]
            reason = cls["reason"]
            level = cls["confidence"]

        frame_rows.append({
            "trip_id": tid,
            "request_date": trip["request_date"],
            "pair_index": i,
            "question": q,
            "answer": a,
            "non_med_nec": category in NON_MED_NEC_CATEGORIES,
            "category": category,
            "criterion_id": criterion,
            "criterion_source": (CONCEPTS.get(criterion, {}).get("source_id")
                                 or PRESUMED.get(criterion, {}).get("cite")
                                 or EXCLUSIONS.get(criterion, {}).get("source_id")
                                 or ("CLINICAL_INTAKE_SURVEY" if criterion in SURVEY_ITEMS else "")
                                 or ("BILLING_GUIDE" if criterion in BED_ELEMENTS else "")),
            "evidence": evidence,
            "reason": reason,
            "confidence": level,
        })

    frame = pd.DataFrame(frame_rows)
    trip_frames[tid] = frame
    rows.extend(frame_rows)

qa_flags = pd.DataFrame(rows)

print("trips with a frame:", len(trip_frames))
print("rows across all trips:", f"{len(qa_flags):,}")
print()
if len(qa_flags):
    print(qa_flags["category"].value_counts().to_string())
    print()
    print("non-medical-necessity pairs:",
          f"{int(qa_flags['non_med_nec'].sum()):,}",
          f"({qa_flags['non_med_nec'].mean() * 100:.1f}%)")
    print()
    print(pd.crosstab(qa_flags["category"], qa_flags["confidence"]).to_string())

## 9. Read one trip

In [ ]:
if trip_frames:
    example_id = next(iter(trip_frames))
    example = trip_frames[example_id]
    print("trip:", example_id)
    print()
    print(example[["pair_index", "question", "answer", "non_med_nec", "category",
                   "criterion_id", "confidence"]].to_string(index=False))
    print()
    for r in example[example["non_med_nec"]].to_dict("records"):
        print("-" * 100)
        print(f"[{r['category']} | {r['confidence']}] {r['question']}")
        print(f"  answer:    {r['answer']}")
        print(f"  criterion: {r['criterion_id'] or 'none'}")
        print(f"  evidence:  {r['evidence'] or 'none'}")
        print(f"  reason:    {r['reason']}")

## 10. Trip level summary

In [ ]:
by_trip = pd.DataFrame()
if len(qa_flags):
    by_trip = (qa_flags.groupby(["trip_id", "request_date"])
               .agg(pairs=("pair_index", "size"),
                    non_med_nec_pairs=("non_med_nec", "sum"),
                    supporting_pairs=("category", lambda s: int((s == "supports_necessity").sum())),
                    high_confidence=("confidence", lambda s: int((s == "High").sum())),
                    low_confidence=("confidence", lambda s: int((s == "Low").sum())))
               .reset_index())
    by_trip["non_med_nec_pct"] = (by_trip["non_med_nec_pairs"] / by_trip["pairs"] * 100).round(1)
    by_trip["any_supporting"] = by_trip["supporting_pairs"] > 0
    by_trip = by_trip.sort_values("non_med_nec_pct", ascending=False).reset_index(drop=True)

    print(by_trip.head(25).to_string(index=False))
    print()
    print("trips with no supporting pair at all:",
          f"{int((~by_trip['any_supporting']).sum()):,} of {len(by_trip):,}")
    print()
    print("most frequently flagged questions")
    flagged = (qa_flags[qa_flags["non_med_nec"]]
               .groupby(["question", "category"]).size().rename("pairs")
               .sort_values(ascending=False).head(20).reset_index())
    print(flagged.to_string(index=False))
    print()
    print("criteria cited most often")
    cited = (qa_flags[qa_flags["criterion_id"] != ""]
             .groupby(["criterion_id", "category"]).size().rename("pairs")
             .sort_values(ascending=False).head(20).reset_index())
    print(cited.to_string(index=False))

## 11. Output

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

criteria_frame = pd.DataFrame(
    [{"criterion_id": k, "kind": "concept", "status": c["status"], "source_id": c["source_id"]}
     for k, c in CONCEPTS.items()]
    + [{"criterion_id": k, "kind": "presumed", "status": "cited", "source_id": "BILLING_GUIDE"}
       for k in PRESUMED]
    + [{"criterion_id": k, "kind": "bed element", "status": "cited", "source_id": "BILLING_GUIDE"}
       for k in BED_ELEMENTS]
    + [{"criterion_id": k, "kind": "non-covered", "status": "cited", "source_id": e["source_id"]}
       for k, e in EXCLUSIONS.items()]
    + [{"criterion_id": k, "kind": "survey answer", "status": "cited",
        "source_id": "CLINICAL_INTAKE_SURVEY"} for k in SURVEY_ITEMS])

TABS = [("QA Flags", qa_flags), ("By Trip", by_trip), ("Criteria", criteria_frame)]
WIDTHS = {"question": 55, "answer": 45, "reason": 60, "evidence": 40,
          "criterion_id": 30, "trip_id": 16, "request_date": 14}

wb = Workbook()
wb.remove(wb.active)
for name, frame in TABS:
    ws = wb.create_sheet(name[:31])
    if frame is None or len(frame) == 0:
        ws.append(["no rows"])
        continue
    ws.append([str(c) for c in frame.columns])
    for cell in ws[1]:
        cell.font = Font(name="Calibri", bold=True)
    for rec in frame.itertuples(index=False):
        out = []
        for v in rec:
            if v is None or (isinstance(v, float) and pd.isna(v)):
                out.append("")
            elif isinstance(v, (int, float, bool)):
                out.append(v)
            else:
                out.append(CONTROL_CHARS.sub(" ", str(v))[:32000])
        ws.append(out)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for i, col in enumerate(frame.columns, start=1):
        ws.column_dimensions[get_column_letter(i)].width = WIDTHS.get(str(col), 16)

xlsx_name = f"med_nec_qa_review_{RUN_TS}.xlsx"
local_xlsx = os.path.join(LOCAL_DIR, xlsx_name)
wb.save(local_xlsx)

try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    shutil.copyfile(local_xlsx, os.path.join(OUTPUT_DIR, xlsx_name))
    print("wrote", os.path.join(OUTPUT_DIR, xlsx_name))
except Exception as e:
    print("workspace copy skipped:", e)
    print("local copy at", local_xlsx)

print()
print("trips:", len(trip_frames), "| pairs:", f"{len(qa_flags):,}")
if len(qa_flags):
    print(qa_flags["confidence"].value_counts().to_string())

## 12. Notes on reading the result

`trip_frames[trip_id]` is the frame for a single trip. `qa_flags` is every trip stacked.

`non_med_nec` is true for the administrative, non_covered and insufficient_alone categories. The
three mean different things and are worth separating before drawing a conclusion. An
administrative pair says nothing about the patient. A non_covered pair describes something
guidance names as not establishing necessity. An insufficient_alone pair does describe a real
element, it simply does not carry the determination by itself, and bed confinement is the
common case.

A trip where every pair is flagged is not the same as a trip that fails medical necessity. It
means the order form captured nothing that supports it, which is a documentation finding rather
than a clinical one.

Confidence describes the classification of that one pair, not the trip, and it is the model's
own rating against the rubric in section 4. A low confidence pair is a candidate for review of
the criteria file, since it usually means the form wording does not resemble anything the
criteria describe.

Read confidence alongside the evidence column. A high confidence classification with an empty
evidence quote is worth checking, since the rubric asks for the quote to carry the
classification.